# Stage 2 — Fact Gathering QC

Run each fact gatherer individually on a small sample of signals to inspect outputs.

| Step | Gatherer | What to check |
|------|----------|---------------|
| 0 | Config | Date windows, sample size |
| 1 | Session + data | Row count |
| 2 | WMD data | Wastage, markdown log, item hierarchy |
| 3 | Detect signals | Source signal list |
| 4 | PPG drilldown | Top contributors, WoW changes |
| 5 | Day-of-week | Worst day per store-dept |
| 6 | Chain drift | Chain OSA context per dept |
| 7 | Repeat history | Per-item week history |
| 8 | Tick-and-flick breakdown | TF rate history per store |
| 9 | Waste breakdown | Top departments by waste qty and rate |
| 10 | Full enrich | QC checks (no missing facts) |

In [1]:
ANALYSIS_DATE_STR = '2025-06-14'

# ── Client config (None = base defaults; 'eastofengland' = client override) ────
CLIENT = None   # or 'eastofengland'

QC_STORE_ID       = None    # set to int to restrict to one store
SAMPLE_N          = 5       # number of signals to use for individual gatherer checks

import datetime
analysis_date   = datetime.date.fromisoformat(ANALYSIS_DATE_STR)
THIS_WEEK_END   = str(analysis_date)
THIS_WEEK_START = str(analysis_date - datetime.timedelta(days=6))
LAST_WEEK_END   = str(analysis_date - datetime.timedelta(days=7))
LAST_WEEK_START = str(analysis_date - datetime.timedelta(days=13))
LOOKBACK_START  = str(analysis_date - datetime.timedelta(days=69))

print(f'This week : {THIS_WEEK_START} – {THIS_WEEK_END}')
print(f'Last week : {LAST_WEEK_START} – {LAST_WEEK_END}')
print(f'Lookback  : {LOOKBACK_START} – {THIS_WEEK_END}')
print(f'Client    : {CLIENT or "base defaults"}')

This week : 2025-06-08 – 2025-06-14
Last week : 2025-06-01 – 2025-06-07
Lookback  : 2025-04-06 – 2025-06-14
Client    : base defaults


In [3]:
import sys, os, logging, warnings

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

# ── Suppress noisy Snowflake connector output ─────────────────────────────────

from store_insight.pipeline.config.loader import load_config
from store_insight.pipeline.pipeline.session import create_session
from snowflake.snowpark import functions as F

cfg = load_config(CLIENT)
print(f'Config loaded: client={cfg["client_name"]}')

session = create_session(cfg)   # prints 'Snowflake session created.'

# ── Resolve table names from config ──────────────────────────────────────────
OSA_TABLE       = cfg['tables']['osa']
HIERARCHY_TABLE = cfg['tables']['hierarchy']
FEEDBACK_TABLE  = cfg['tables']['feedback']

print(f'OSA table       : {OSA_TABLE}')
print(f'Hierarchy table : {HIERARCHY_TABLE}')
print(f'Feedback table  : {FEEDBACK_TABLE}')

osa_raw = (
    session.table(OSA_TABLE)
    .filter(F.col('ANALYSIS_DATE') >= LOOKBACK_START)
    .filter(F.col('ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('SITE_ID').alias('STORE_ID'), F.col('ITEM_ID').alias('ITEM_CODE'),
        F.col('ACTUAL_SALES_VOLUME'), F.col('LOST_SALES_VOLUME'),
        F.col('LOST_SALES_VALUE'), F.col('ANALYSIS_DATE'), F.col('DEPARTMENT_ID'),
    )
)
item = (
    session.table(HIERARCHY_TABLE)
    .select(
        F.col('MASTER_ITEM_ID').alias('ITEM_CODE'), F.col('ITEM_ID'),
        F.col('DEPARTMENT_NAME'), F.col('SUBCLASS_NAME'), F.col('PRODUCT_PROFILE_GROUP_NAME'),
    ).distinct()
)
osa = (
    osa_raw.join(item, on=['ITEM_CODE'], how='left')
    .filter(F.col('DEPARTMENT_NAME').is_not_null())
    .cache_result()
)

feedback = (
    session.table(FEEDBACK_TABLE)
    .filter(F.col('ALERT_ANALYSIS_DATE') >= LAST_WEEK_START)
    .filter(F.col('ALERT_ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('SITE_ID').alias('STORE_ID'),
        F.col('SITE_NAME'),
        F.col('ALERT_TYPE_NAME'),
        F.col('ALERT_ANALYSIS_DATE'),
        F.col('INTERVENTION_ID'),
        F.col('TICK_AND_FLICK_FLAG'),
    )
    .cache_result()
)

qc_data = osa.filter(F.col('STORE_ID') == QC_STORE_ID) if QC_STORE_ID else osa
print(f'\nOSA rows cached     : {osa.count():,}')
print(f'Feedback rows cached: {feedback.count():,}')

Config loaded: client=default


 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/93a31d5d-1a5d-433f-baa0-138549cd2108/saml2?SAMLRequest=nZJNc9owEIb%2Fikc925Zsk4IGyJAwtExISoPp101YgmiQJVcrxUl%2FfYWJZ9JDcujF45GfXT%2Fad8eXT7WKHoUFafQEkQSjSOjKcKkPE7QtF%2FEQReCY5kwZLSboWQC6nI6B1aqhM%2B8e9L347QW4KDTSQLsPE%2BStpoaBBKpZLYC6im5mtyuaJZg21jhTGYVelbxfwQCEdcGwL%2BEgg96Dcw1N07ZtkzZPjD2kGcY4xaM0UCfkQ88%2FhTu9wZMUFyc%2BEAFfv7hdSX0ewXtauzME9HNZruP1l02Jolmvem00%2BFrYjbCPshLb%2B9VZAIIB1OQix0XijzGYMMOE%2FfFWJKBNu1fsKCpTN96F1kl4S%2FeCp8ocZLj9cj5BzVHyYc7LDezY5nD33WxnvpmvWPbxZtTeHq4%2B%2BcWN%2BVlqR9iPr9WyQtG3Pt7sFO8SwIulPoXqwhHOLmJcxGRQ4hHFOS1wEuR%2BoWgeQpWaua6yN%2B88klpW1oDZO6OV1KKzHOUsJ3zAY8LCo8jzfbxjDMckHw6KUcUzgofpKeoMndeHdiJ2%2Bl9DGaevW7ys411IaDlfGyWr52hhbM3c2wGShHQnksf7DqWiZlLNOLcCIASplGmvrWAubL2zXqB0ev7r

 pip install snowflake-connector-python[secure-local-storage]


Snowflake session created.
OSA table       : ANALYSIS.FINAL.LSM_ITEM_STORE_DAY_REPORTING_PERIOD
Hierarchy table : ANALYSIS.FINAL.LSM_ITEM_HIERARCHY
Feedback table  : ANALYSIS.FINAL.LSM_ALERT_FEEDBACK

OSA rows cached     : 73,588,943
Feedback rows cached: 22,275


In [ ]:
from snowflake.snowpark import Window
from snowflake.snowpark.types import LongType

MARKDOWN_LOG_TABLE    = cfg['tables'].get('markdown_log',        'ANALYSIS.FINAL.WMD_MARKDOWN_API')
MARKDOWN_SALES_TABLE  = cfg['tables'].get('markdown_sales',      'ANALYSIS.FINAL.WMD_SALES')
WASTAGE_TABLE         = cfg['tables'].get('wastage',             'ANALYSIS.FINAL.WMD_WASTAGE')
WMD_ITEM_HIER_TABLE   = cfg['tables'].get('wmd_item_hierarchy',  'ANALYSIS.FINAL.WMD_ITEM_HIERARCHY')
WMD_STORE_HIER_TABLE  = cfg['tables'].get('wmd_store_hierarchy', 'ANALYSIS.FINAL.WMD_SITE_HIERARCHY')

WMD_LOOKBACK = str(analysis_date - datetime.timedelta(days=13))  # two full weeks

# WMD_MARKDOWN_API: SITE_ID→STORE_ID, MARKDOWN_DATE→LOCAL_DATE, MARKDOWN_QUANTITY→QTY_MARKDOWN
markdown_log = (
    session.table(MARKDOWN_LOG_TABLE)
    .with_column('STORE_ID',     F.col('SITE_ID'))
    .with_column('LOCAL_DATE',   F.col('MARKDOWN_DATE'))
    .with_column('QTY_MARKDOWN', F.col('MARKDOWN_QUANTITY'))
    .filter(F.col('LOCAL_DATE') >= WMD_LOOKBACK)
    .filter(F.col('LOCAL_DATE') <= THIS_WEEK_END)
    .cache_result()
)
# WMD_SALES: SITE_ID→STORE_ID, SALE_QUANTITY→MARKDOWN_QUANTITY, SALE_VALUE→MARKDOWN_VALUE
markdown_sales = (
    session.table(MARKDOWN_SALES_TABLE)
    .with_column('STORE_ID',          F.col('SITE_ID'))
    .with_column('MARKDOWN_QUANTITY', F.col('SALE_QUANTITY'))
    .with_column('MARKDOWN_VALUE',    F.col('SALE_VALUE'))
    .filter(F.col('DATE') >= WMD_LOOKBACK)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .cache_result()
)
# WMD_WASTAGE: SITE_ID→STORE_ID, WASTE_DATE→DATE
wastage = (
    session.table(WASTAGE_TABLE)
    .with_column('STORE_ID', F.col('SITE_ID'))
    .with_column('DATE',     F.col('WASTE_DATE'))
    .filter(F.col('DATE') >= WMD_LOOKBACK)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .cache_result()
)

# WMD_ITEM_HIERARCHY: VME_CODE→ITEM_ID, ITEM_NAME→ITEM_DESCRIPTION, HIERARCHY_LEVEL_1_ID→DEPARTMENT
_item_w = Window.partitionBy('VME_CODE').orderBy(F.col('FILE_TIMESTAMP').desc())
item_hierarchy = (
    session.table(WMD_ITEM_HIER_TABLE)
    .with_column('_RN', F.row_number().over(_item_w))
    .filter(F.col('_RN') == 1)
    .select(
        F.col('VME_CODE').cast(LongType()).alias('ITEM_ID'),
        F.col('ITEM_NAME').alias('ITEM_DESCRIPTION'),
        F.col('HIERARCHY_LEVEL_1_ID').alias('DEPARTMENT'),
        F.col('SECTION'),
        F.col('SUB_SECTION'),
    )
    .filter(F.col('ITEM_ID').is_not_null())
    .cache_result()
)
# WMD_SITE_HIERARCHY: SITE_ID→STORE_ID, SITE_NAME→STORE_NAME
wmd_store_hierarchy = (
    session.table(WMD_STORE_HIER_TABLE)
    .with_column('STORE_ID',   F.col('SITE_ID'))
    .with_column('STORE_NAME', F.col('SITE_NAME'))
    .cache_result()
)

print(f'Markdown log rows  : {markdown_log.count():,}')
print(f'Markdown sales rows: {markdown_sales.count():,}')
print(f'Wastage rows       : {wastage.count():,}')
print(f'Item hierarchy rows (deduplicated): {item_hierarchy.count():,}')
print(f'Store hierarchy rows: {wmd_store_hierarchy.count():,}')

Markdown log rows  : 127,541
Markdown sales rows: 324,605
Wastage rows       : 22,940
Item hierarchy rows (deduplicated): 179,706
Store hierarchy rows: 97,250


In [5]:
from store_insight.pipeline.stages._1_detect.detect import detect_signals

signals = detect_signals(
    qc_data,
    last_week_start=LAST_WEEK_START, last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START, this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    feedback=feedback,
    wastage=wastage,
    markdown_log=markdown_log,
    markdown_sales=markdown_sales,
    item_hierarchy=item_hierarchy,
    store_hierarchy=wmd_store_hierarchy,
    cfg=cfg,
)

osa_signals   = [s for s in signals if s.get('signal_type') == 'OSA']
waste_signals = [s for s in signals if s.get('signal_type') == 'WASTE']
sample        = osa_signals[:SAMPLE_N]

print(f'Total signals: {len(signals)}  '
      f'(OSA={len(osa_signals)}, WASTE={len(waste_signals)})')
for s in sample:
    lsv = s.get('this_week_lsv', 0) or 0
    print(f"  store={s['store_id']:>4}  dept={s.get('department', ''):<30}  "
          f"triggers={s['trigger_types']}  lsv=£{lsv:,.2f}")

[1_detect] TICK_AND_FLICK=13 (chain avg 35.4%)
[1_detect] MARKDOWN_OVERRIDE VS_CHAIN=13 (chain avg 4.2%)
[1_detect] MARKDOWN_OVERRIDE WOW=6
[1_detect] MARKDOWN_SELLTHROUGH VS_CHAIN=56 (chain avg 99.1%)
[1_detect] MARKDOWN_SELLTHROUGH WOW=5
[1_detect] MARKDOWN_SKU EXCESSIVE_OVERRIDE=390 (lookback=4w, min_rate=50%)
[1_detect] MARKDOWN_SKU REPEAT_REQUEST=4203 (lookback=4w, min_weeks=3)
[1_detect] WASTE VS_CHAIN=27 (chain avg 14.1%)
[1_detect] WASTE WOW=32
[1_detect] 5208 signals — TICK_AND_FLICK=13, VS_CHAIN=27, WOW=32, EXCESSIVE_OVERRIDE=390, REPEAT_REQUEST=4203, REPEAT=321
Total signals: 5208  (OSA=574, WASTE=37)
  store= 505  dept=TOBACCO                         triggers=['REPEAT']  lsv=£987.12
  store= 234  dept=BEER/LAGER & CIDER              triggers=['REPEAT', 'VS_CHAIN']  lsv=£541.44
  store= 215  dept=CIGARETTES                      triggers=['REPEAT']  lsv=£1,620.92
  store= 270  dept=FRESH BEEF                      triggers=['REPEAT']  lsv=£107.10
  store= 445  dept=CIGARETTES 

## Step 3 — PPG Drilldown

In [5]:
from store_insight.pipeline.stages._2_facts.gatherers.osa_ppg_drilldown import gather as gather_ppg

ppg_data = gather_ppg(
    osa, sample,
    last_week_start=LAST_WEEK_START, last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START, this_week_end=THIS_WEEK_END,
)

for (store_id, dept), d in ppg_data.items():
    print(f'\n  store={store_id}  dept={dept}')
    print('  Top contributors (by this-week LSV):')
    for sc in d['subclass_contributors']:
        print(f"    • {sc['subclass_name']:<35} TW=£{sc['this_week_lsv']:>8,.2f}  "
              f"LW=£{sc['last_week_lsv']:>8,.2f}  chg={sc['lsv_change']:+,.2f}")
    print('  Biggest WoW increases:')
    for sc in d['subclass_changes']:
        print(f"    • {sc['subclass_name']:<35} chg={sc['lsv_change']:+,.2f}")


  store=447  dept=CIGARETTES
  Top contributors (by this-week LSV):
    • SUPERKINGS MILD/LIGHTS              TW=£  230.83  LW=£  256.20  chg=-25.37
    • KINGSIZE STANDARD                   TW=£   85.92  LW=£  159.84  chg=-73.91
    • KINGSIZE MILD/LIGHTS                TW=£   70.83  LW=£   69.35  chg=+1.48
    • SUPERKINGS STANDARD                 TW=£   27.37  LW=£   17.74  chg=+9.63
    • KINGSIZE ULTRA                      TW=£    0.00  LW=£    0.00  chg=+0.00
  Biggest WoW increases:
    • SUPERKINGS STANDARD                 chg=+9.63
    • KINGSIZE MILD/LIGHTS                chg=+1.48
    • KINGSIZE ULTRA                      chg=+0.00
    • SUPERKINGS MILD/LIGHTS              chg=-25.37
    • KINGSIZE STANDARD                   chg=-73.91

  store=530  dept=CIGARETTES
  Top contributors (by this-week LSV):
    • KINGSIZE MILD/LIGHTS                TW=£  183.28  LW=£  211.34  chg=-28.06
    • SUPERKINGS STANDARD                 TW=£   86.93  LW=£  117.19  chg=-30.26
    • KINGS

## Step 4 — Day-of-Week Pattern

In [6]:
from store_insight.pipeline.stages._2_facts.gatherers.osa_dow import gather as gather_dow

dow_data = gather_dow(osa, sample, this_week_start=THIS_WEEK_START, this_week_end=THIS_WEEK_END)

for (store_id, dept), days in dow_data.items():
    print(f'\n  store={store_id}  dept={dept}')
    max_lsv = max(d['lsv'] for d in days) or 1
    for d in days:
        bar = '█' * int(d['lsv'] / max_lsv * 20)
        print(f"    {d['day']:<10} £{d['lsv']:>8,.2f}  {bar}")


  store=530  dept=CIGARETTES
    Friday     £   97.96  ████████████████████
    Monday     £   78.56  ████████████████
    Thursday   £   71.67  ██████████████
    Tuesday    £   67.45  █████████████
    Wednesday  £   50.26  ██████████
    Saturday   £   42.58  ████████
    Sunday     £   28.31  █████

  store=462  dept=EGGS
    Wednesday  £   14.68  ████████████████████
    Sunday     £    6.84  █████████
    Tuesday    £    6.53  ████████
    Thursday   £    4.58  ██████
    Monday     £    1.27  █
    Friday     £    0.00  
    Saturday   £    0.00  

  store=497  dept=TOBACCO
    Saturday   £  118.79  ████████████████████
    Wednesday  £   69.23  ███████████
    Sunday     £   34.25  █████
    Tuesday    £   24.88  ████
    Thursday   £   11.77  █
    Monday     £    7.46  █
    Friday     £    2.83  

  store=460  dept=CIGARETTES
    Monday     £  140.25  ████████████████████
    Wednesday  £  131.99  ██████████████████
    Sunday     £  108.51  ███████████████
    Tuesday    £

## Step 5 — Chain Drift

In [7]:
from store_insight.pipeline.stages._2_facts.gatherers.osa_chain_drift import gather as gather_chain

# Always pass the full osa DataFrame (not qc_data) for chain averages
chain_data = gather_chain(
    osa, sample,
    last_week_start=LAST_WEEK_START, last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START, this_week_end=THIS_WEEK_END,
)

print(f'Chain drift for {len(chain_data)} departments:')
for dept, c in chain_data.items():
    print(f"  {dept:<30}  LW={c['chain_last_week_osa']:.1f}%  "
          f"TW={c['chain_this_week_osa']:.1f}%  drift={c['chain_drift_pp']:+.2f}pp")

Chain drift for 3 departments:
  TOBACCO                         LW=91.5%  TW=93.4%  drift=+1.89pp
  CIGARETTES                      LW=89.3%  TW=91.2%  drift=+1.82pp
  EGGS                            LW=91.7%  TW=94.5%  drift=+2.86pp


## Step 6 — Repeat Item History

In [8]:
from store_insight.pipeline.stages._2_facts.gatherers.osa_repeat_history import gather as gather_history

repeat_sample = [s for s in signals if 'REPEAT' in s['trigger_types']][:3]
print(f'REPEAT signals in sample: {len(repeat_sample)}')

hist_data = gather_history(osa, repeat_sample, lookback_start=LOOKBACK_START, lookback_end=THIS_WEEK_END)

for (store_id, dept), items in hist_data.items():
    print(f'\n  store={store_id}  dept={dept}  ({len(items)} items)')
    for item_code, weeks in list(items.items())[:2]:
        print(f'    item={item_code}  {len(weeks)} weeks of history')
        for w in weeks[-4:]:  # last 4 weeks
            print(f"      yr={w['year']} wk={w['week']:>2}  osa={w['weekly_osa']:.1f}%  lsv=£{w['weekly_lsv']:,.2f}")

REPEAT signals in sample: 3

  store=530  dept=CIGARETTES  (9 items)
    item=751241  11 weeks of history
      yr=2025 wk=21  osa=100.0%  lsv=£0.00
      yr=2025 wk=22  osa=43.5%  lsv=£180.06
      yr=2025 wk=23  osa=55.6%  lsv=£141.10
      yr=2025 wk=24  osa=74.3%  lsv=£65.46
    item=546  11 weeks of history
      yr=2025 wk=21  osa=72.2%  lsv=£14.29
      yr=2025 wk=22  osa=100.0%  lsv=£0.00
      yr=2025 wk=23  osa=0.0%  lsv=£0.00
      yr=2025 wk=24  osa=44.9%  lsv=£22.78

  store=460  dept=CIGARETTES  (9 items)
    item=751241  11 weeks of history
      yr=2025 wk=21  osa=62.5%  lsv=£30.26
      yr=2025 wk=22  osa=86.2%  lsv=£16.09
      yr=2025 wk=23  osa=84.5%  lsv=£27.75
      yr=2025 wk=24  osa=0.0%  lsv=£89.80
    item=32957  11 weeks of history
      yr=2025 wk=21  osa=39.6%  lsv=£57.44
      yr=2025 wk=22  osa=63.2%  lsv=£29.32
      yr=2025 wk=23  osa=100.0%  lsv=£0.00
      yr=2025 wk=24  osa=31.7%  lsv=£27.04

  store=447  dept=CIGARETTES  (9 items)
    item=794153  1

## Step 7 — Tick-and-Flick Breakdown

In [9]:
from store_insight.pipeline.stages._2_facts.gatherers.tick_and_flick_breakdown import gather as gather_tf

tf_signals = [s for s in signals if 'TICK_AND_FLICK' in s['trigger_types']]
print(f'TICK_AND_FLICK signals in full set: {len(tf_signals)}')

tf_sample = tf_signals[:3]
tf_data = gather_tf(feedback, tf_sample,
                    this_week_start=THIS_WEEK_START, this_week_end=THIS_WEEK_END)

for store_id, d in tf_data.items():
    print(f"\n  store={store_id:>4}")
    for row in d.get('tf_by_alert_type', []):
        print(f"    alert_type={row['alert_type']:<30}  worked={row['worked']:>4}  "
              f"ticked={row['tf_count']:>4}  tf_rate={row['tf_rate']:.1f}%")

TICK_AND_FLICK signals in full set: 13

  store= 407
    alert_type=Availability                    worked=  96  ticked=  52  tf_rate=54.2%

  store= 217
    alert_type=Availability                    worked= 109  ticked=  65  tf_rate=59.6%

  store= 448
    alert_type=Availability                    worked=  96  ticked=  59  tf_rate=61.5%


## Step 9 — Waste Department Breakdown

In [10]:
from store_insight.pipeline.stages._2_facts.gatherers.waste_dept_breakdown import gather as gather_waste

print(f'WASTE signals available: {len(waste_signals)}')

waste_sample = waste_signals[:3]
waste_data = gather_waste(
    wastage, markdown_log, waste_sample, item_hierarchy,
    this_week_start=THIS_WEEK_START, this_week_end=THIS_WEEK_END,
)

for store_id, d in waste_data.items():
    sig = next(s for s in waste_sample if s['store_id'] == store_id)
    print(f"\n  store={store_id:>4}  {sig.get('site_name', ''):<25}  "
          f"waste_rate={sig.get('this_week_stb_rate', 0):.1f}%  "
          f"chain={sig.get('chain_stb_rate', 0):.1f}%  "
          f"above=+{sig.get('above_chain_pp', 0):.1f}pp")
    for row in d['waste_dept_breakdown']:
        print(f"    {row['dept']:<30}  waste_qty={row['waste_qty']:>5}  "
              f"rate={row['waste_rate_pct']:.1f}%  val=£{row['waste_value']:,.2f}")

WASTE signals available: 37

  store= 492  Norwich Station            waste_rate=67.2%  chain=14.1%  above=+53.1pp
    DELICATESSEN                    waste_qty=   72  rate=0.0%  val=£199.04
    PROVISIONS PREPARED             waste_qty=   26  rate=0.0%  val=£56.64
    SOFT BEVERAGES                  waste_qty=    9  rate=0.0%  val=£15.11
    GROC 3 - SNACKING               waste_qty=    7  rate=0.0%  val=£10.33
    FRUIT & VEGETABLES              waste_qty=    5  rate=0.0%  val=£15.15

  store= 516  East Bergholt Old          waste_rate=30.2%  chain=14.1%  above=+16.2pp
    FRUIT & VEGETABLES              waste_qty=  157  rate=0.0%  val=£263.92
    BAKERY                          waste_qty=   41  rate=0.0%  val=£70.57
    PROVISIONS PREPARED             waste_qty=   35  rate=0.0%  val=£88.74
    MEAT & POULTRY                  waste_qty=   24  rate=0.0%  val=£112.86
    PROVISIONS DAIRY                waste_qty=   24  rate=0.0%  val=£42.52

  store= 633  Bishops Stortford          was

## Step 8 — Full Enrich (QC checks)

In [11]:
from store_insight.pipeline.stages._2_facts.facts import enrich_signals

enriched = enrich_signals(
    signals, osa,
    last_week_start=LAST_WEEK_START, last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START, this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    feedback=feedback,
    wastage=wastage,
    markdown_log=markdown_log,
    item_hierarchy=item_hierarchy,
    cfg=cfg,
)

osa_enriched    = [s for s in enriched if s.get('signal_type') == 'OSA']
tf_enriched     = [s for s in enriched if s.get('signal_type') == 'COMPLIANCE']
waste_enriched  = [s for s in enriched if s.get('signal_type') == 'WASTE']

missing_chain  = [s for s in osa_enriched if s['facts'].get('chain_this_week_osa') is None]
missing_ppg    = [s for s in osa_enriched if not s['facts'].get('subclass_contributors')]
missing_dow    = [s for s in osa_enriched if not s['facts'].get('dow_pattern')]
missing_tf     = [s for s in tf_enriched  if not s['facts'].get('tf_by_alert_type')]
missing_waste  = [s for s in waste_enriched if not s['facts'].get('waste_dept_breakdown')]

print(f'{len(enriched)} enriched signals  '
      f'(OSA={len(osa_enriched)}, COMPLIANCE={len(tf_enriched)}, WASTE={len(waste_enriched)})')
print(f'QC OSA:   missing chain_this_week_osa  : {len(missing_chain)}')
print(f'QC OSA:   missing subclass_contributors: {len(missing_ppg)}')
print(f'QC OSA:   missing dow_pattern          : {len(missing_dow)}')
print(f'QC TF:    missing tf_by_alert_type     : {len(missing_tf)}')
print(f'QC WASTE: missing waste_dept_breakdown : {len(missing_waste)}')

print('\n--- OSA spot-check (first 3) ---')
for sig in osa_enriched[:3]:
    f = sig['facts']
    print(f"\n  store={sig['store_id']:>4}  dept={sig['department']:<30}  triggers={sig['trigger_types']}")
    print(f"    lsv=£{sig.get('this_week_lsv', 0):,.2f}  cost_projection=£{f.get('cost_projection', 0):,.2f}")
    print(f"    chain_osa={f.get('chain_this_week_osa')}%  worst_day={f.get('worst_day')}")
    print(f"    top_ppg={f['subclass_contributors'][0]['subclass_name'] if f.get('subclass_contributors') else 'n/a'}")

print('\n--- WASTE spot-check (first 3) ---')
for sig in waste_enriched[:3]:
    f = sig['facts']
    rows = f.get('waste_dept_breakdown', [])
    print(f"\n  store={sig['store_id']:>4}  {sig.get('site_name', ''):<25}  "
          f"waste_rate={sig.get('this_week_stb_rate', 0):.1f}%  chain={sig.get('chain_stb_rate', 0):.1f}%")
    for row in rows[:3]:
        print(f"    {row['dept']:<30}  waste_qty={row['waste_qty']:>5}  "
              f"rate={row['waste_rate_pct']:.1f}%  val=£{row['waste_value']:,.2f}")

print('\n--- COMPLIANCE spot-check (first 3) ---')
for sig in tf_enriched[:3]:
    f = sig['facts']
    rows = f.get('tf_by_alert_type', [])
    print(f"\n  store={sig['store_id']:>4}  ({len(rows)} alert types)")
    for row in rows[:5]:
        print(f"    {row['alert_type']:<30}  worked={row['worked']:>4}  "
              f"ticked={row['tf_count']:>4}  tf_rate={row['tf_rate']:.1f}%")

AttributeError: 'NoneType' object has no attribute 'filter'